In [3]:

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# check if a GPU is available and set the device accordingly for faster training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# define a pipeline to resize images to 64x64 and convert them into PyTorch Tensors
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# load the dataset from a folder structure-Tiny ImageNet expects folders to be class names
dataset = torchvision.datasets.ImageFolder(
    root='./tiny-imagenet/train',   # MAKE SURE PATH IS CORRECT
    transform=transform
)

# output basic metadata about the loaded dataset
print("Total images:", len(dataset))
print("Classes:", dataset.classes)

# function to separate a specific class (the "anomaly") from the rest of the data
def split_anomaly_dataset(dataset, anomaly_class_idx=9):
    # to store indices of "normal" data
    seen_indices = []

    # to store indices of "anomalous" data
    unseen_indices = []

    # iterate through the entire dataset to check labels
    for i in range(len(dataset)):
        # get the image (ignored) and its class index
        _, label = dataset[i]
        if label == anomaly_class_idx:
            # add to anomaly list if it matches the target index
            unseen_indices.append(i)
        else:
            # add to seen/normal list otherwise
            seen_indices.append(i)

    # create Subset objects which act as filtered versions of the original dataset
    seen_dataset = Subset(dataset, seen_indices)
    unseen_dataset = Subset(dataset, unseen_indices)

    return seen_dataset, unseen_dataset

# execute the split using index 9 as the unseen/anomaly class
seen_data, unseen_data = split_anomaly_dataset(dataset, anomaly_class_idx=9)

# verify the size of the resulting splits
print("Seen samples:", len(seen_data))
print("Unseen samples:", len(unseen_data))

# define how many images to process at once
batch_size = 64

# seen_loader: used for training
seen_loader = DataLoader(seen_data, batch_size=batch_size, shuffle=True)
# unseen_loader: used for testing/evaluation
unseen_loader = DataLoader(unseen_data, batch_size=batch_size, shuffle=False)

Using device: cpu
Total images: 3500
Classes: ['n07871810', 'n07873807', 'n07875152', 'n07920052', 'n09193705', 'n09246464', 'n09256479', 'n09332890', 'n09428293', 'n12267677']
Seen samples: 3150
Unseen samples: 350


In [4]:
# define the Convolutional Autoencoder class inheriting from nn.Module
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        
        # the Encoder
        self.encoder = nn.Sequential(

            # Input: 3 channels (RGB), Output: 16 filters, Kernel: 3x3, Stride: 2
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.ReLU(),

            # Input: 16, Output: 32 filters
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),

            # Input: 32, Output: 64 filters
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU()
        )
    
        # The Decoder
        self.decoder = nn.Sequential(

            # Transposed Convolution
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            # Upsamples from 32 filters to 16
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),

            # Final layer: returns to 3 channels (RGB)
            nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    # define the data flow through the model
    def forward(self, x):
        # pass input through encoder
        encoded = self.encoder(x)

        # pass compressed version through decoder
        decoded = self.decoder(encoded)

        # return the reconstructed image
        return decoded


# initialize the model and move it to the GPU/CPU
cae_model = ConvAutoencoder().to(device)

# Mean Squared Error loss
criterion = nn.MSELoss()
# Adam optimizer
optimizer = torch.optim.Adam(cae_model.parameters(), lr=1e-3)


epochs = 10  

# Training Loop
for epoch in range(epochs):
    # set model to training mode
    cae_model.train()

    # track cumulative loss for the epoch
    running_loss = 0.0

    # iterate through batches from the 'seen' data loader
    for images, _ in seen_loader:
        # move image batch to GPU/CPU
        images = images.to(device)

        # forward pass: get reconstructed images
        outputs = cae_model(images)
        # calculate loss: compare reconstruction (outputs) to original (images)
        loss = criterion(outputs, images)

        # Backward pass and optimization ->

        # cear previous gradients
        optimizer.zero_grad()

        # compute gradients via backpropagation
        loss.backward()

        # update weights
        optimizer.step()

        # add batch loss to the total
        running_loss += loss.item()

    # print the average loss per epoch to monitor progress
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss:.4f}")


Epoch [1/10] Loss: 2.8864
Epoch [2/10] Loss: 1.4460
Epoch [3/10] Loss: 1.0638
Epoch [4/10] Loss: 0.8583
Epoch [5/10] Loss: 0.7328
Epoch [6/10] Loss: 0.6942
Epoch [7/10] Loss: 0.6629
Epoch [8/10] Loss: 0.6426
Epoch [9/10] Loss: 0.6339
Epoch [10/10] Loss: 0.6207


In [5]:
# function to calculate the average reconstruction error (MSE) for a given dataset
def evaluate_reconstruction_error(model, dataloader, device):
    # set model to evaluation mode
    model.eval()

    # re-initialize the loss function for measuring error
    criterion = nn.MSELoss()
    
    # cumulative loss across all batches
    total_loss = 0.0
    # counter for total number of images processed
    total_samples = 0
    
    # disable gradient calculation to save memory and speed up computation during inference
    with torch.no_grad():
        for images, _ in dataloader:
            # move data to the active device (GPU or CPU)
            images = images.to(device)
            
            # forward pass
            outputs = model(images)
            # calculate error between original and reconstruction
            loss = criterion(outputs, images)
            
            # total_loss update
            total_loss += loss.item() * images.size(0)
            # keep track of the total count of images
            total_samples += images.size(0)
            
    # calculate the global average MSE across all images in the loader
    average_mse = total_loss / total_samples
    return average_mse


# run evaluation on the training/normal data
print("\nEvaluating Seen Classes...")
seen_mse = evaluate_reconstruction_error(cae_model, seen_loader, device)

# run evaluation on the held-out anomaly class
print("Evaluating Unseen (Anomaly) Class...")
unseen_mse = evaluate_reconstruction_error(cae_model, unseen_loader, device)

print("-" * 30)
print(f"Average MSE for Seen Classes:   {seen_mse:.6f}")
print(f"Average MSE for Unseen Class:   {unseen_mse:.6f}")
print(f"Difference:                    {unseen_mse - seen_mse:.6f}")


Evaluating Seen Classes...
Evaluating Unseen (Anomaly) Class...
------------------------------
Average MSE for Seen Classes:   0.012419
Average MSE for Unseen Class:   0.014441
Difference:                    0.002022


In [6]:

# ==============================
# PART 3: ABLATION (Sigmoid vs Tanh final layer)
# ==============================

activations = {
    "Sigmoid": nn.Sigmoid(),
    "Tanh": nn.Tanh()
}

ablation_results = {}

for name, final_act in activations.items():
    model = ConvAutoencoder().to(device)
    model.decoder[-1] = final_act  # explicitly replace output layer activation

    print(f"\nRunning with output layer = {name}")
    print(f"Final decoder layer now: {model.decoder[-1]}")

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for _ in range(epochs):
        model.train()
        for images, _ in seen_loader:
            images = images.to(device)
            outputs = model(images)
            loss = criterion(outputs, images)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    seen_mse = evaluate_reconstruction_error(model, seen_loader, device)
    unseen_mse = evaluate_reconstruction_error(model, unseen_loader, device)
    gap = unseen_mse - seen_mse

    ablation_results[name] = {
        "seen_mse": seen_mse,
        "unseen_mse": unseen_mse,
        "gap": gap
    }

    print(f"Seen MSE:   {seen_mse:.6f}")
    print(f"Unseen MSE: {unseen_mse:.6f}")
    print(f"Gap:        {gap:.6f}")

print("\n" + "=" * 45)
print("Comparison Summary")
print("=" * 45)

for name, metrics in ablation_results.items():
    print(f"{name:8s} | Seen: {metrics['seen_mse']:.6f} | Unseen: {metrics['unseen_mse']:.6f} | Gap: {metrics['gap']:.6f}")

best_seen = min(ablation_results, key=lambda k: ablation_results[k]["seen_mse"])
best_gap = max(ablation_results, key=lambda k: ablation_results[k]["gap"])

print(f"\nLower reconstruction loss on seen data: {best_seen}")
print(f"Higher anomaly separation (gap):       {best_gap}")


Running with output layer = Sigmoid
Final decoder layer now: Sigmoid()
Seen MSE:   0.012627
Unseen MSE: 0.014822
Gap:        0.002194

Running with output layer = Tanh
Final decoder layer now: Tanh()
Seen MSE:   0.011721
Unseen MSE: 0.014098
Gap:        0.002378

Comparison Summary
Sigmoid  | Seen: 0.012627 | Unseen: 0.014822 | Gap: 0.002194
Tanh     | Seen: 0.011721 | Unseen: 0.014098 | Gap: 0.002378

Lower reconstruction loss on seen data: Tanh
Higher anomaly separation (gap):       Tanh
